In [ ]:
# ==============================================================================
# CELDA 1: LIBRERÍAS Y PARÁMETROS CERTIFICADOS (BINARIO - TRIAL 10)
# ==============================================================================
import os, random, re, glob, warnings, copy, gc, cv2, types
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

import scipy.stats as st
import shap  

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, confusion_matrix, cohen_kappa_score, accuracy_score,
                             f1_score, roc_auc_score, roc_curve, precision_recall_curve, auc, log_loss)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import BayesianRidge, LogisticRegression

from torch.optim.swa_utils import AveragedModel, SWALR

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")


from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, auc, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

 
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict


import os
import random
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# Parámetros
DEBUG_MODE = False    
SEED = 13          
N_SPLITS = 5       
BATCH_SIZE = 4      
ACCUM_STEPS = 4     # Batch Efectivo = 16  
N_CORTES = 16      

# Hiperparámetros
LR_BACKBONE = 2.79e-05
LR_HEAD = 4.20e-05
WEIGHT_DECAY = 3.57e-03
CUTMIX_PROB = 0.2207
LABEL_SMOOTH = 0.1117
SWA_LR_OPT = 6.23e-05
FOCAL_GAMMA = 1.1793

if DEBUG_MODE:
    N_SPLITS = 2; EPOCHS_RUN = 3; SWA_START = 1; ACCUM_STEPS = 1
    print(" DEBUG_MODE ACTIVO.")
else:
    EPOCHS_RUN = 40; SWA_START = 25

# Inicialización de semillas
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Aceleración: {dispositivo}")

# AUMENTO MORFOLÓGICO (Simulación de Atrofia Ex-Vacuo)
 class MorphologicalAugmentation:
    def __init__(self, p=0.3, kernel_size=3):
        self.p = p; self.kernel = np.ones((kernel_size, kernel_size), np.uint8)
    def __call__(self, img):
        if random.random() < self.p:
            img_np = np.array(img)
            img_np = cv2.dilate(img_np, self.kernel, iterations=1) if random.random() < 0.5 else cv2.erode(img_np, self.kernel, iterations=1)
            return Image.fromarray(img_np)
        return img

In [ ]:
# ==============================================================================
# CELDA 2: CARGA CLÍNICA, MAPEADO BINARIO ESTRICTO Y MUESTREO DEBUG
# ==============================================================================
PATH_IMGS = "/kaggle/input/datasets/ninadaithal/imagesoasis"
EXCEL_FILES = glob.glob("/kaggle/input/**/*.xlsx", recursive=True)

df_demog = pd.read_excel(EXCEL_FILES[0])
df_demog['id_paciente'] = df_demog['ID'].str.extract(r'(OAS1_\d{4})')
df_demog = df_demog.drop_duplicates(subset=['id_paciente'], keep='first')

# Filtrado por viabilidad (Edad >= 56)
valid_ids = set(df_demog[df_demog['Age'] >= 56]['id_paciente'].dropna().unique())

archivos = []
for root, _, files in os.walk(PATH_IMGS):
    for f in files:
        if f.lower().endswith(('.png', '.jpg')):
            pid_match = re.search(r"OAS1_\d{4}", f, re.IGNORECASE)
            if pid_match and pid_match.group(0).upper() in valid_ids:
                archivos.append({"ruta": os.path.join(root, f), "id": pid_match.group(0).upper()})

df_raw = pd.DataFrame(archivos)
df_pacientes = df_raw[['id', 'ruta']].drop_duplicates(subset=['id']).reset_index(drop=True)

cols_clinicas = ['id_paciente', 'Age', 'Educ', 'SES', 'MMSE', 'eTIV', 'nWBV', 'CDR']
df_pacientes = pd.merge(df_pacientes, df_demog[cols_clinicas], left_on='id', right_on='id_paciente', how='inner')

#  MAPEO BINARIO ESTRICTO (CDR == 0 -> Sano [0], CDR > 0 -> Alzheimer [1])
df_pacientes = df_pacientes.dropna(subset=['CDR']).reset_index(drop=True) # Purga de NaNs Clínicos
df_pacientes['etiqueta'] = (df_pacientes['CDR'] > 0).astype(int)

# Estratificación Dual
df_pacientes['Age_Quartile'] = pd.qcut(df_pacientes['Age'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
df_pacientes['strat_dual'] = df_pacientes['etiqueta'].astype(str) + "_" + df_pacientes['Age_Quartile'].astype(str)

conteo = df_pacientes['strat_dual'].value_counts()
clases_min = conteo[conteo < N_SPLITS].index
df_pacientes.loc[df_pacientes['strat_dual'].isin(clases_min), 'strat_dual'] = df_pacientes.loc[df_pacientes['strat_dual'].isin(clases_min), 'etiqueta'].astype(str)

# ---------------------------------------------------------
# INTERVENCIÓN DE DEBUG
# ---------------------------------------------------------
if DEBUG_MODE:
    print("\n MODO DEBUG ACTIVADO\n")
    muestras_por_estrato = max(N_SPLITS, (BATCH_SIZE * ACCUM_STEPS) // len(df_pacientes['strat_dual'].unique()) + 1)
    df_pacientes = df_pacientes.groupby('strat_dual', group_keys=False).apply(
        lambda x: x.sample(min(len(x), muestras_por_estrato), random_state=SEED)
    ).reset_index(drop=True)

print(f" Dataset Binario. N Total Pacientes: {len(df_pacientes)}")

In [ ]:
# ==============================================================================
# CELDA 3: DATASET VISUAL PRO (EXTRACCIÓN LINSPACE 2.5D)
# ==============================================================================
class OASIS_Visual_Dataset(Dataset):
    """
    Motor de alimentación de datos (Dataloader) personalizado para PyTorch.
    Transforma rutas de carpetas de resonancias magnéticas en Tensores 2.5D.
    Implementa un muestreo espacial mediante np.linspace para descartar ruido
    anatómico (cráneo/cuello) y aislar el volumen cerebral útil.
    """
    def __init__(self, dataframe, is_train=True):
        # 1. INICIALIZACIÓN Y MAPEADO DE DATOS
        self.rutas = dataframe['ruta'].values        # Rutas a las carpetas de cada paciente
        self.targets = dataframe['etiqueta'].values  # Etiquetas diagnósticas (0, 0.5, 1...)
        self.pids = dataframe['id'].values           # Identificadores únicos (ej. OAS1_0001)
        self.is_train = is_train                     # Booleano: True para Entrenamiento, False para Test/Validación
        
        # 2. TRANSFORMACIONES BASE (Obligatorias para ConvNeXt)
        self.transform_base = transforms.Compose([
            transforms.Resize((224, 224)),           # ConvNeXt requiere tensores espaciales de 224x224
            transforms.ToTensor(),                   # Convierte píxeles (0-255) a tensores matemáticos (0.0-1.0)
        ])
        
        # 3. AUMENTO DE DATOS (Regularización Estocástica para evitar Overfitting)
        self.augment = transforms.Compose([
            MorphologicalAugmentation(p=0.3),        # Simulación de atrofia/ensanchamiento ventricular (30% prob)
            transforms.RandomRotation(5),            # Rotación sutil (±5°) simulando movimiento del paciente
            # Traslación (2%) y escalado (0.98-1.02) para forzar invarianza espacial en la red
            transforms.RandomAffine(degrees=0, translate=(0.02, 0.02), scale=(0.98, 1.02))
        ])

    def __len__(self): 
        # Devuelve el tamaño total de la cohorte para el cálculo de épocas
        return len(self.rutas)
        
    def __getitem__(self, idx):
        # 4. LOCALIZACIÓN DEL PACIENTE
        paciente_id = self.pids[idx]
        carpeta_clase = os.path.dirname(self.rutas[idx])
        
        # 5. LECTURA Y ORDENACIÓN ANATÓMICA
        todos = os.listdir(carpeta_clase)
        # CRÍTICO: Se ordenan alfabéticamente/numéricamente para mantener la secuencia 
        # anatómica real (eje axial) y no mezclar cortes físicos del cerebro.
        archivos_paciente = sorted([f for f in todos if f.lower().endswith(('.png', '.jpg')) and paciente_id.lower() in f.lower()])
        
        # 6. HEURÍSTICA DE RECORTE (Bounding Box Z-Axis)
        total = len(archivos_paciente)
        # Descartamos el 20% inicial (base del cuello) y el 25% superior (bóveda craneal pura)
        inicio, fin = int(total * 0.20), int(total * 0.75)
        
        # 7. EXTRACCIÓN 2.5D (Linspace Sampling)
        # Selecciona exactamente 'N_CORTES' (ej. 16) distribuidos uniformemente dentro del rango útil.
        # Esto captura la profundidad ventricular sin saturar la VRAM de la GPU.
        indices = np.linspace(inicio, max(inicio, fin - 1), N_CORTES).astype(int)
        
        cortes_tensor = []
        # 8. PROCESAMIENTO DE CADA CORTE INDIVIDUAL
        for i in indices:
            # ConvNeXt exige 3 canales (RGB), aunque la RM sea en escala de grises
            img = Image.open(os.path.join(carpeta_clase, archivos_paciente[i])).convert('RGB')
            
            # Aplicamos aumentos estocásticos SOLO si estamos en fase de entrenamiento
            if self.is_train: 
                img = self.augment(img)
                
            t = self.transform_base(img)
            
            # 9. NORMALIZACIÓN Z-SCORE A NIVEL DE INSTANCIA
            # Mitiga el sesgo de contraste entre diferentes escáneres hospitalarios
            # (media 0, varianza 1). El 1e-6 previene la división por cero en cortes oscuros.
            t = (t - t.mean()) / (t.std() + 1e-6) 
            
            cortes_tensor.append(t)
            
        # 10. APILADO FINAL 2.5D
        # Apila los cortes en una nueva dimensión. Forma final: [N_CORTES, 3, 224, 224]
        return torch.stack(cortes_tensor), torch.tensor(self.targets[idx]).float()

print(" Motor Dataset listo.")

In [ ]:
# ==============================================================================
# CELDA 4: ARQUITECTURA SOTA BINARIA (CONVNEXT + SPD-CONV)
# ==============================================================================

class SPDConv(nn.Module):
    """
    Space-to-Depth Convolution. Un bloque de reducción de dimensionalidad que 
    evita el Max-Pooling tradicional para no destruir información anatómica de subpíxel.
    """
    def __init__(self, in_channels, out_channels, block_size=2):
        super().__init__()
        self.block_size = block_size
        # Al mover píxeles espaciales a canales, los canales de entrada se multiplican por block_size^2 (ej. x4)
        self.conv = nn.Conv2d(in_channels * (block_size ** 2), out_channels, kernel_size=3, padding=1, bias=False)
        self.norm = nn.LayerNorm(out_channels)
        
    def forward(self, x):
        B, C, H, W = x.size() # Batch, Canales, Alto, Ancho
        bs = self.block_size
        
        # 1. Padding dinámico: Asegura que la imagen sea perfectamente divisible por el tamaño del bloque (2x2)
        pad_h, pad_w = (bs - H % bs) % bs, (bs - W % bs) % bs
        if pad_h > 0 or pad_w > 0: 
            x = F.pad(x, (0, pad_w, 0, pad_h))
            H, W = H + pad_h, W + pad_w
            
        out_H, out_W = H // bs, W // bs
        
        # 2. Reestructuración topológica (La magia del Space-to-Depth)
        # Descompone el espacio, permuta las dimensiones y lo aplasta hacia el eje de canales.
        x = x.view(B, C, out_H, bs, out_W, bs).permute(0, 1, 3, 5, 2, 4).contiguous().view(B, C * (bs ** 2), out_H, out_W)
        
        # 3. Convolución de fusión + Normalización (Permutando para que LayerNorm funcione en la dimensión correcta)
        return self.norm(self.conv(x).permute(0, 2, 3, 1)).permute(0, 3, 1, 2)


class MAF(nn.Module):
    """
    Mixed Activation Function. Suaviza la topología del gradiente al promediar
    tres de las funciones de activación más avanzadas.
    """
    def forward(self, x): 
        return (F.relu(x) + F.silu(x) + F.gelu(x)) / 3.0


class HybridAgile_MIDL(nn.Module):
    """
    Arquitectura Híbrida 2.5D. 
    Integra un backbone moderno (ConvNeXt), atención entre cortes (Slice-Attention) 
    y compresión latente para diagnóstico clínico.
    """
    def __init__(self, num_slices=16, feature_dim=768, hidden_dim=256):
        super().__init__()
        
        # 1. EL BACKBONE (Motor visual base)
        self.backbone = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        
        # Congelamiento parcial (LLRD setup para el modelo binario)
        # Solo congelamos los bloques 0 y 1. Los bloques más profundos podrán aprender.
        for param in self.backbone.features[0:2].parameters(): 
            param.requires_grad = False
            
        # Anulamos el clasificador original de ImageNet (No queremos clasificar perros o gatos)
        self.backbone.classifier = nn.Identity()
        
        # 2. EXTRACTOR FINO Y REDUCCIÓN
        self.fine_grained_extractor = SPDConv(in_channels=768, out_channels=feature_dim, block_size=2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1)) # Global Average Pooling (GAP)
        
        # 3. MECANISMO DE ATENCIÓN (Slice-Attention)
        # Calcula qué cortes (slices) contienen la información clínica más relevante
        self.attention_V = nn.Sequential(nn.Linear(feature_dim, hidden_dim), MAF())
        self.attention_U = nn.Sequential(nn.Linear(feature_dim, hidden_dim), nn.Sigmoid())
        self.attention_weights = nn.Linear(hidden_dim, 1) # Salida 1: Un score por cada corte
        
        # 4. ESPACIO LATENTE Y CLASIFICADOR FINAL
        # Dropout muy agresivo (0.4) para evitar la memorización con pocos pacientes
        self.latent_extractor = nn.Sequential(nn.Linear(feature_dim, 128), nn.LayerNorm(128), MAF(), nn.Dropout(p=0.4))
        self.classifier = nn.Linear(128, 1) # Un solo nodo de salida (Predicción Binaria: 0 a 1)

    def forward(self, x):
        # x tiene forma 5D: [Batch, Slices, Channels, Alto, Ancho] -> ej: [4, 16, 3, 224, 224]
        B, S, C, H, W = x.shape
        
        # TRUCO 2.5D: Fusionamos Batch y Slices para engañar al backbone 2D
        x = x.view(B * S, C, H, W) 
        
        # Extracción de mapas de características visuales
        features_spatial = self.backbone.features(x) 
        
        # Preservación subpíxel y Global Average Pooling
        features_flat = self.adaptive_pool(self.fine_grained_extractor(features_spatial)).view(B * S, -1)
        
        # CÁLCULO DE ATENCIÓN
        # Multiplicamos ramas V y U y obtenemos un mapa de pesos normalizado (Softmax)
        A = F.softmax(self.attention_weights(self.attention_V(features_flat) * self.attention_U(features_flat)).view(B, S), dim=1)
        
        # Deshacemos el truco 2.5D para separar los pacientes de sus cortes
        features_flat = features_flat.view(B, S, -1)
        
        # FUSIÓN PONDERADA (Batch Matrix Multiplication)
        # Multiplica cada vector por su "puntuación de atención" y los suma en un vector único por paciente
        M = torch.bmm(A.unsqueeze(1), features_flat).squeeze(1)
        
        # Proyección al Espacio Latente de 128 dimensiones
        latent_128d = self.latent_extractor(M)
        
        # Devuelve: Logits (Predicción final), el Espacio Latente (para explicabilidad) y los pesos de atención (para ver qué corte importó)
        return self.classifier(latent_128d), latent_128d, A

print(" Arquitectura ConvNeXt instanciada correctamente.")

In [ ]:
# ==============================================================================
# CELDA 5: MOTOR VISUAL BINARIO SOTA (GUARDA PESOS EN DISCO PARA XAI)
# ==============================================================================

DIR_GUARDADO = './modelos_entrenados/'
os.makedirs(DIR_GUARDADO, exist_ok=True) # Crea la carpeta si no existe para guardar los pesos de la red

# Transformaciones afines (rotación/escala) usadas para el Test-Time Augmentation (TTA)
tta_transforms_test = transforms.Compose([transforms.RandomAffine(degrees=5, scale=(0.9, 1.0))])
# N_TTA es 10 en entrenamiento real (1 inferencia original + 9 alteradas)
N_TTA = 10 if not DEBUG_MODE else 2

resultados_visuales_por_fold = {}
historial_train_loss, historial_val_loss = [], []

# Instanciamos el K-Fold para particionar a nivel de paciente
skf_outer = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

def cutmix_data(x, y, alpha=1.0):
    """
    Mezcla dos imágenes y sus etiquetas en proporción geométrica.
    x: Tensor de imágenes de entrada
    y: Tensor de etiquetas clínicas
    """
    # Genera la proporción de mezcla usando una distribución Beta
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    # Mezcla el batch actual consigo mismo de forma aleatoria
    index = torch.randperm(x.size(0)).to(x.device)
    
    W, H = x.size(-1), x.size(-2)
    # Calcula el tamaño del parche rectangular a recortar
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    
    # Selecciona un centro aleatorio para el parche
    cx, cy = np.random.randint(W), np.random.randint(H)
    
    # Calcula los límites del bounding box
    bbx1, bby1 = np.clip(cx - cut_w // 2, 0, W), np.clip(cy - cut_h // 2, 0, H)
    bbx2, bby2 = np.clip(cx + cut_w // 2, 0, W), np.clip(cy + cut_h // 2, 0, H)
    
    # Inserta el parche del paciente B en el paciente A
    x[:, :, :, bby1:bby2, bbx1:bbx2] = x[index, :, :, bby1:bby2, bbx1:bbx2]
    
    # Ajusta lambda según el área real recortada
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (W * H))
    
    # Devuelve: Imagen mezclada, etiqueta A, etiqueta B, y el porcentaje de mezcla (lam)
    return x, y, y[index], lam

class FocalSmoothLoss(nn.Module):
    """
    Pérdida Híbrida: Focal Loss (contra el desbalance) + Smooth Loss (agrupación en el espacio latente)
    """
    def __init__(self, alpha=0.25, gamma=FOCAL_GAMMA, lambda_fs=0.05, label_smoothing=LABEL_SMOOTH):
        super().__init__()
        # Usamos BCE sin reducción para poder multiplicar por los pesos focales luego
        self.bce = nn.BCEWithLogitsLoss(reduction='none')
        self.alpha = alpha; self.gamma = gamma; self.lambda_fs = lambda_fs; self.ls = label_smoothing

    def forward(self, logits, latents, targets):
        logits = logits.float(); latents = latents.float(); targets = targets.float()
        
        # 1. Label Smoothing: Convierte [0, 1] en [0.05, 0.95] para evitar certidumbre absoluta
        targets_smooth = targets * (1.0 - self.ls) + 0.5 * self.ls
        
        # 2. Focal Loss
        bce_loss = self.bce(logits, targets_smooth)
        pt = torch.exp(-bce_loss) # Probabilidad de acierto real
        # Multiplicamos el BCE por (1-pt)^gamma. Si pt es alto (fácil), el castigo tiende a 0.
        f_loss = (self.alpha * (1 - pt)**self.gamma * bce_loss).mean()
        
        # 3. Feature Smoothing Loss (Contrastiva Intraclase)
        fs_loss = torch.tensor(0.0, device=logits.device)
        for c in torch.unique(targets): # Para cada clase (0 o 1) en el batch
            mask = (targets == c).squeeze()
            class_latents = latents[mask]
            # Si hay más de 1 paciente de esa clase en el batch...
            if class_latents.dim() > 1 and class_latents.size(0) > 1:
                centroid = class_latents.mean(dim=0, keepdim=True) # Calcula su centro de gravedad
                # Añade pérdida si los pacientes están muy lejos de su propio centroide
                fs_loss += torch.mean(torch.sum((class_latents - centroid)**2, dim=1))
                
        return f_loss + self.lambda_fs * fs_loss

def evaluar_fold_binario_local(y_true, y_prob):
    """ Evalúa métricas buscando el umbral óptimo (Youden Index) """
    if len(np.unique(y_true)) < 2: return None
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    opt_idx = np.argmax(tpr - fpr) # Índice de Youden (Maximiza Sensibilidad y Especificidad a la vez)
    opt_th = thresholds[opt_idx]
    
    y_pred = (y_prob >= opt_th).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    
    return {
        'auc': roc_auc_score(y_true, y_prob), 'pr_auc': auc(rec, prec),
        'acc': accuracy_score(y_true, y_pred),
        'sens': tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        'esp': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'f1': f1_score(y_true, y_pred, zero_division=0)
    }

# ==========================================
# BUCLE PRINCIPAL DE ENTRENAMIENTO K-FOLD
# ==========================================
for fold_out, (train_val_idx, test_idx) in enumerate(skf_outer.split(df_pacientes, df_pacientes['strat_dual'])):
    print(f"\n FOLD {fold_out + 1}/{N_SPLITS}")
    
    # Separación Train+Val y Test
    df_train_val = df_pacientes.iloc[train_val_idx].copy()
    df_test_outer = df_pacientes.iloc[test_idx].copy()
    
    # Prevención de fallos en K-Fold interno si hay clases minoritarias extremas
    min_clase = df_train_val['strat_dual'].value_counts().min()
    if min_clase < 2:
        indices_tr = np.arange(len(df_train_val)); np.random.shuffle(indices_tr)
        split_idx = max(1, int(len(indices_tr) * 0.8))
        train_idx_inner, val_idx_inner = indices_tr[:split_idx], indices_tr[split_idx:]
    else:
        skf_split = StratifiedKFold(n_splits=min(5, min_clase), shuffle=True, random_state=SEED) 
        train_idx_inner, val_idx_inner = next(skf_split.split(df_train_val, df_train_val['strat_dual']))
    
    # Datasets (usando pin_memory=True para acelerar transferencia CPU->GPU)
    loader_tr = DataLoader(OASIS_Visual_Dataset(df_train_val.iloc[train_idx_inner], is_train=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    loader_va = DataLoader(OASIS_Visual_Dataset(df_train_val.iloc[val_idx_inner], is_train=False), batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
    loader_te = DataLoader(OASIS_Visual_Dataset(df_test_outer, is_train=False), batch_size=BATCH_SIZE, num_workers=2, pin_memory=True) 

    # Instanciación de modelos y herramientas
    model = HybridAgile_MIDL().to(dispositivo)
    swa_model = AveragedModel(model) # Modelo secundario que acumula la media de los pesos
    scaler_amp = torch.cuda.amp.GradScaler() # Escala gradientes para usar Precisión Mixta (AMP)
    crit_clasificacion = FocalSmoothLoss(label_smoothing=LABEL_SMOOTH).to(dispositivo)
    
    # IMPLEMENTACIÓN DEL LLRD (Layer-wise Learning Rate Decay)
    m = model.module if isinstance(model, nn.DataParallel) else model
    # Parámetros de la cabecera final (Aprende rápido: LR_HEAD)
    head_params = list(m.fine_grained_extractor.parameters()) + list(m.attention_V.parameters()) + list(m.attention_U.parameters()) + list(m.attention_weights.parameters()) + list(m.latent_extractor.parameters()) + list(m.classifier.parameters())
    # Parámetros del extractor ConvNeXt (Aprende lento: LR_BACKBONE)
    stage4_params = list(m.backbone.features[6].parameters()) + list(m.backbone.features[7].parameters())
    stage3_params = list(m.backbone.features[4].parameters()) + list(m.backbone.features[5].parameters())
    stage2_params = list(m.backbone.features[2].parameters()) + list(m.backbone.features[3].parameters())
    
    for p in head_params + stage4_params + stage3_params + stage2_params: p.requires_grad = True
        
    # Asignación de tasas en el Optimizador AdamW
    param_groups = [{'params': head_params, 'lr': LR_HEAD}, {'params': stage4_params + stage3_params + stage2_params, 'lr': LR_BACKBONE}]
    opt = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY) 
    
    # Schedulers (Estrategias de enfriamiento de la Tasa de Aprendizaje)
    pasos = max(1, len(loader_tr) // ACCUM_STEPS)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=pasos*10, T_mult=2, eta_min=1e-6)
    swa_scheduler = SWALR(opt, swa_lr=SWA_LR_OPT)
    
    best_swa_loss, best_swa_state = float('inf'), None
    fold_t_loss, fold_v_loss = [], []
    
    for epoch in range(EPOCHS_RUN):
        model.train(); t_loss = 0.0; opt.zero_grad()
        
        # BUCLE INTERNO (Batches)
        for i, (inputs, labels) in enumerate(loader_tr):
            # non_blocking=True paraleliza la transferencia a GPU
            inputs, labels = inputs.to(dispositivo, dtype=torch.float32, non_blocking=True), labels.to(dispositivo, dtype=torch.float32).unsqueeze(1)
            
            # Contexto de Precisión Mixta (autocast): Convierte tensores a Float16 para no saturar la GPU
            with torch.cuda.amp.autocast():
                # Aplicamos CutMix solo antes de que empiece SWA (para no desestabilizar la media final)
                if (epoch < SWA_START) and (np.random.rand() < CUTMIX_PROB):
                    inputs, labels_a, labels_b, lam = cutmix_data(inputs, labels)
                    logits, latents, _ = model(inputs)
                    # Pérdida ponderada según el % de mezcla
                    loss = (lam * crit_clasificacion(logits, latents, labels_a) + (1 - lam) * crit_clasificacion(logits, latents, labels_b)) / ACCUM_STEPS
                else:
                    logits, latents, _ = model(inputs)
                    loss = crit_clasificacion(logits, latents, labels) / ACCUM_STEPS # Divide la loss para simular un Batch más grande
                    
            # Retropropagación escalada (AMP)
            scaler_amp.scale(loss).backward()
            
            # ACUMULACIÓN DE GRADIENTES
            # Solo actualiza los pesos cada ACCUM_STEPS. Si Batch=4 y Accum=4, la red aprende como si el Batch fuera 16.
            if (i + 1) % ACCUM_STEPS == 0 or (i + 1) == len(loader_tr):
                scaler_amp.unscale_(opt)
                # Clip Norm para evitar "Explosión de Gradientes"
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler_amp.step(opt); scaler_amp.update(); opt.zero_grad(set_to_none=True)
                
                # Paso del Scheduler
                if epoch >= SWA_START: swa_scheduler.step()
                else: scheduler.step()
                
            if not torch.isnan(loss): t_loss += loss.item() * ACCUM_STEPS
            
        # -------------------------------------
        # FASE DE EVALUACIÓN (Al final de la época)
        # -------------------------------------
        model.eval(); v_loss_total = 0.0
        v_preds_epoch, v_targets_epoch = [], []
        with torch.no_grad():
            for imgs, targets in loader_va:
                imgs, targets_t = imgs.to(dispositivo, non_blocking=True), targets.to(dispositivo, dtype=torch.float32).unsqueeze(1)
                with torch.cuda.amp.autocast():
                    logits, latents, _ = model(imgs) 
                    v_loss_total += crit_clasificacion(logits, latents, targets_t).item()
                    v_preds_epoch.extend(torch.sigmoid(logits.float()).cpu().numpy().flatten())
                    v_targets_epoch.extend(targets.numpy().flatten())
        
        fold_t_loss.append(t_loss / max(1, len(loader_tr)))
        fold_v_loss.append(v_loss_total / max(1, len(loader_va)))
        
        try: auc_ep = roc_auc_score(v_targets_epoch, v_preds_epoch)
        except ValueError: auc_ep = 0.5 
            
        estado_swa = f"(CutMix {CUTMIX_PROB} Activo)" if epoch < SWA_START else ""
        
        # -------------------------------------
        # FASE SWA (Stochastic Weight Averaging)
        # -------------------------------------
        if epoch >= SWA_START:
            # Añade los pesos actuales al modelo promedio
            swa_model.update_parameters(model); swa_model.eval(); swa_loss_val = 0.0
            swa_preds_epoch = []
            with torch.no_grad():
                # Evalúa el rendimiento del modelo promedio (suele ser mejor y más estable)
                for imgs, targets in loader_va:
                    imgs, targets_t = imgs.to(dispositivo, non_blocking=True), targets.to(dispositivo, dtype=torch.float32).unsqueeze(1)
                    with torch.cuda.amp.autocast(): 
                        logits, latents, _ = swa_model(imgs)
                        swa_loss_val += crit_clasificacion(logits, latents, targets_t).item()
                        swa_preds_epoch.extend(torch.sigmoid(logits.float()).cpu().numpy().flatten())
            swa_loss_val /= max(1, len(loader_va))
            
            try: swa_auc_ep = roc_auc_score(v_targets_epoch, swa_preds_epoch)
            except ValueError: swa_auc_ep = 0.5
                
            estado_swa = f"(SWA | Val Loss: {swa_loss_val:.4f} | SWA AUC: {swa_auc_ep:.4f})"
            
            # Guardamos el mejor modelo SWA
            if swa_loss_val < best_swa_loss: 
                best_swa_loss = swa_loss_val
                best_swa_state = copy.deepcopy(swa_model.state_dict())

        print(f"    Ep {epoch+1:02d}/{EPOCHS_RUN} | Train L: {fold_t_loss[-1]:.4f} | Val L: {fold_v_loss[-1]:.4f} | Val AUC: {auc_ep:.4f} {estado_swa}")

    # Restaura el mejor modelo promediado de SWA antes del Test
    if best_swa_state is not None: swa_model.load_state_dict(best_swa_state)
    historial_train_loss.append(fold_t_loss); historial_val_loss.append(fold_v_loss)

    # ==========================================
    # INFERENCIA TTA (Test-Time Augmentation)
    # ==========================================
    def inferir_dual_sota(loader):
        ps_all, ts_all = [], []
        for imgs, targets in loader:
            imgs = imgs.to(dispositivo, non_blocking=True); B, S, C, H, W = imgs.shape; swa_model.eval() 
            with torch.no_grad(), torch.cuda.amp.autocast():
                # Inferencia 1: Imagen Original
                logits_base, _, _ = swa_model(imgs) 
                probs_det = [torch.nan_to_num(torch.sigmoid(logits_base.float()))]
                
                # Inferencia 2 a 10: Imágenes Alteradas Afínmente
                for _ in range(N_TTA - 1): 
                    logits_tta = swa_model(tta_transforms_test(imgs.view(B*S, C, H, W)).view(B, S, C, H, W))[0]
                    probs_det.append(torch.nan_to_num(torch.sigmoid(logits_tta.float())))
                    
                # Promedio de todas las predicciones  
                p_final = torch.stack(probs_det).mean(dim=0)
            ps_all.extend(p_final.cpu().numpy().flatten()); ts_all.extend(targets.numpy().flatten())
        return np.array(ps_all), np.array(ts_all)

    print("    Calculando Inferencia Final TTA (Test Set)...")
    t_preds_raw, t_targets = inferir_dual_sota(loader_te)
    
    # Evaluación con Youden
    res_fold = evaluar_fold_binario_local(t_targets, t_preds_raw)
    if res_fold:
        print(f"     TEST AUC: {res_fold['auc']:.4f} | PR-AUC: {res_fold['pr_auc']:.4f} | Sens: {res_fold['sens']:.4f} | Esp: {res_fold['esp']:.4f}")
        
    resultados_visuales_por_fold[fold_out] = {
        'test_preds': t_preds_raw, 'test_targets': t_targets, 'test_idx': np.array(test_idx)
    }
    
    # Guardamos los pesos en disco para poder extraer mapas Grad-CAM más adelante
    ruta_pesos = os.path.join(DIR_GUARDADO, f'swa_model_fold_{fold_out}.pth')
    torch.save(swa_model.module.state_dict(), ruta_pesos)
    
    # LIMPIEZA DE VRAM (Evita errores CUDA Out of Memory en Folds posteriores)
    del model, swa_model, loader_tr, loader_va, loader_te; gc.collect(); torch.cuda.empty_cache()

# Guardamos las curvas de aprendizaje en disco para poder imprimirlas en la Celda 7
ruta_npz = os.path.join(DIR_GUARDADO, 'resultados_visuales_oof.npz')
np.savez_compressed(ruta_npz, train_loss_curve=np.mean(historial_train_loss, axis=0), val_loss_curve=np.mean(historial_val_loss, axis=0))
print(f"\n Entrenamiento Visual completado. Resultados y pesos guardados en disco.")

In [ ]:
# ==============================================================================
# CELDA 6: FUSIÓN BINARIA SOTA (LASSO META-LEARNER / SELECCIÓN DE RASGOS)
# ==============================================================================
print("="*80)
print(" Régimen Binario")
print("="*80)

# -------------------------------------------------------------------------
# 1. EVALUACIÓN Y MÉTRICAS (Índice de Youden)
# -------------------------------------------------------------------------
def evaluar_fold_binario(y_true, y_prob):
    if len(np.unique(y_true)) < 2: return None
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    opt_idx = np.argmax(tpr - fpr)
    opt_th = thresholds[opt_idx]
    
    y_pred = (y_prob >= opt_th).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    
    return {
        'auc': roc_auc_score(y_true, y_prob), 'pr_auc': auc(rec, prec),
        'acc': accuracy_score(y_true, y_pred),
        'sens': tp / (tp + fn) if (tp + fn) > 0 else 0.0, 
        'esp': tn / (tn + fp) if (tn + fp) > 0 else 0.0, 
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'cm': cm, 'y_true': y_true, 'y_prob': y_prob, 
        'fpr': fpr, 'tpr': tpr, 'precision': prec, 'recall': rec
    }

def safe_logit(p):
    """Proyecta probabilidades [0,1] al espacio continuo Logit [-inf, +inf]."""
    p_clipped = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p_clipped / (1.0 - p_clipped))

# -------------------------------------------------------------------------
# 2. PURGA DIMENSIONAL Y EXTRACCIÓN OOF VISUAL
# -------------------------------------------------------------------------
if 'M/F' not in df_pacientes.columns and 'M/F' in df_demog.columns:
    df_pacientes = pd.merge(df_pacientes, df_demog[['id_paciente', 'M/F']], left_on='id', right_on='id_paciente', how='left')

if 'M/F' in df_pacientes.columns:
    df_pacientes['M/F_bin'] = df_pacientes['M/F'].map({'M': 1, 'F': 0}).fillna(0)
    cols_reserva = ['Age', 'Educ', 'SES', 'eTIV', 'M/F_bin']
else:
    cols_reserva = ['Age', 'Educ', 'SES', 'eTIV']

df_pacientes['Prob_Visual_RM'] = np.nan
for fold_idx, data in resultados_visuales_por_fold.items():
    if isinstance(data, np.ndarray): data = data.item()
    df_pacientes.loc[data['test_idx'], 'Prob_Visual_RM'] = data['test_preds']

if df_pacientes['Prob_Visual_RM'].isna().sum() > 0:
    df_pacientes['Prob_Visual_RM'].fillna(df_pacientes['Prob_Visual_RM'].median(), inplace=True)

# -------------------------------------------------------------------------
# 3. ESCENARIOS CLÍNICOS
# -------------------------------------------------------------------------
escenarios = [
    ('2. Tabular (Sin MMSE, Sin nWBV)', cols_reserva),
    ('3. Tabular (Sin MMSE)',           cols_reserva + ['nWBV']),
    ('4. Tabular (Completo)',           cols_reserva + ['MMSE', 'nWBV']),
    ('5. Fusión (Sin MMSE, Sin nWBV)',  cols_reserva + ['Prob_Visual_RM']),
    ('6. Fusión Profunda (Sin MMSE)',   cols_reserva + ['nWBV', 'Prob_Visual_RM']),
    ('7. Fusión Profunda (Completo)',   cols_reserva + ['MMSE', 'nWBV', 'Prob_Visual_RM'])
]

orden_deseado = ["1. Visual Aislada"] + [e[0] for e in escenarios]
almacen_resultados = {m: [] for m in orden_deseado}
registro_alphas = {m: [] for m in orden_deseado if 'Fusión' in m}

for fold_idx, data in resultados_visuales_por_fold.items():
    if isinstance(data, np.ndarray): data = data.item()
    r = evaluar_fold_binario(data['test_targets'], data['test_preds'])
    if r: almacen_resultados["1. Visual Aislada"].append(r)

# -------------------------------------------------------------------------
# 4. BUCLE MULTIMODAL CON LASSO META-LEARNER
# -------------------------------------------------------------------------
for nombre_modelo, cols in escenarios:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    is_fusion = 'Prob_Visual_RM' in cols
    cols_tab = [c for c in cols if c != 'Prob_Visual_RM']
    
    for tr_idx, te_idx in skf.split(df_pacientes, df_pacientes['strat_dual']):
        df_tr, df_te = df_pacientes.iloc[tr_idx].copy(), df_pacientes.iloc[te_idx].copy()
        y_tr, y_te = df_tr['etiqueta'].values, df_te['etiqueta'].values
        
        imputer = SimpleImputer(strategy='median')
        scaler = StandardScaler()
        X_tr_sc = scaler.fit_transform(imputer.fit_transform(df_tr[cols_tab]))
        X_te_sc = scaler.transform(imputer.transform(df_te[cols_tab]))
        
        modelo_tabular = LogisticRegression(penalty='l2', C=0.5, class_weight='balanced', random_state=SEED)
        modelo_tabular.fit(X_tr_sc, y_tr)
        p_tab_te = modelo_tabular.predict_proba(X_te_sc)[:, 1]
        
        if is_fusion:
            cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
            p_tab_tr_oof = cross_val_predict(modelo_tabular, X_tr_sc, y_tr, cv=cv_inner, method='predict_proba')[:, 1]
            
            p_vis_tr = df_tr['Prob_Visual_RM'].values
            p_vis_te = df_te['Prob_Visual_RM'].values
            
            l_vis_tr, l_tab_tr = safe_logit(p_vis_tr), safe_logit(p_tab_tr_oof)
            l_vis_te, l_tab_te = safe_logit(p_vis_te), safe_logit(p_tab_te)
            
            meta_scaler = StandardScaler()
            X_meta_tr = meta_scaler.fit_transform(np.column_stack((l_vis_tr, l_tab_tr)))
            X_meta_te = meta_scaler.transform(np.column_stack((l_vis_te, l_tab_te)))
            
            meta_juez = LogisticRegressionCV(
                Cs=10, 
                cv=3, 
                penalty='l1', 
                solver='liblinear', 
                scoring='roc_auc', 
                class_weight='balanced',
                random_state=SEED
            )
            meta_juez.fit(X_meta_tr, y_tr)
            
            preds_te = meta_juez.predict_proba(X_meta_te)[:, 1]
            
            w_v, w_t = np.abs(meta_juez.coef_[0])
            alpha_virtual = w_v / (w_v + w_t + 1e-7)
            registro_alphas[nombre_modelo].append(alpha_virtual)
        else:
            preds_te = p_tab_te
            
        res = evaluar_fold_binario(y_te, preds_te)
        if res: almacen_resultados[nombre_modelo].append(res)

# -------------------------------------------------------------------------
# 5. REPORTE FINAL
# -------------------------------------------------------------------------
def calc_ci(data):
    m = np.mean(data); err = st.sem(data)
    if err == 0 or len(data) < 2: return m, m, m, 0.0
    h = err * st.t.ppf((1 + 0.95) / 2., len(data)-1)
    return m, max(0, m-h), min(1, m+h), h

print("==========================================================================================================")
print(" Reportes del rendimiento de los distintos modelos (BINARIO)")
print("==========================================================================================================")
for modelo in orden_deseado:
    print(f"\n MODELO: {modelo.upper()}")
    if modelo in registro_alphas:
        mean_alpha = np.mean(registro_alphas[modelo])
        print(f"   [!] Lasso Stacking Weight (Importancia Real): α = {mean_alpha:.3f} ({mean_alpha*100:.1f}% Imagen / {(1-mean_alpha)*100:.1f}% Tabla)")
        
    df_m = pd.DataFrame(columns=["Métrica", "Media", "95% CI", "Variación"])
    for i, (k, name) in enumerate({'auc': 'AUC ROC', 'pr_auc': 'PR-AUC', 'f1': 'F1-Score', 'sens': 'Sensibilidad', 'esp': 'Especificidad'}.items()):
        datos = [r[k] for r in almacen_resultados[modelo]]
        if len(datos) > 0:
            med, inf, sup, var = calc_ci(datos)
            df_m.loc[i] = [name, f"{med:.4f}", f"[{inf:.4f}-{sup:.4f}]", f"±{var:.4f}"]
        else:
            df_m.loc[i] = [name, "N/A", "N/A", "N/A"]
    print(df_m.to_string(index=False))
    print("-" * 100)

In [ ]:
# ==============================================================================
# CELDA 7: BATERÍA GRÁFICA ULTIMATE Q1 PARA MEMORIA DE TFG (BINARIO + SHAP)
# ==============================================================================
DIR_GRAFICOS = './graficos_tfg/'
os.makedirs(DIR_GRAFICOS, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
colores_modelos = ['#9b59b6', '#34495e', '#e74c3c', '#f39c12', '#1abc9c', '#3498db'] # Ajustado a 6 colores

 
orden_filtrado = []
for m in orden_deseado:
    # Destruimos exclusivamente el modelo puramente tabular de demografía
    if 'Tabular' in m and ('Demografía' in m or 'Demografia' in m) and 'Sin MMSE' not in m and 'Volumetría' not in m:
        continue
    if m.startswith('2. Tabular'): # Por si acaso empieza así
        continue
    orden_filtrado.append(m)

def obtener_etiqueta(nombre):
    """Asigna los nombres limpios solicitados asegurando el orden 1 al 6"""
    if '1.' in nombre or ('Visual' in nombre and 'Fusión' not in nombre): 
        return "1 Visual (ConvNeXt)"
    elif '3.' in nombre or ('Tabular' in nombre and 'Sin MMSE' in nombre): 
        return "2 Tabular (Sin MMSE)"
    elif '4.' in nombre or ('Tabular' in nombre and 'Completo' in nombre): 
        return "3 Tabular (Completo)"
    elif '5.' in nombre or ('Fusión' in nombre and 'Demografía' in nombre): 
        return "4 Fusión (Demografía)"
    elif '6.' in nombre or ('Fusión' in nombre and 'Sin MMSE' in nombre): 
        return "5 Fusión (Sin MMSE)"
    elif '7.' in nombre or ('Fusión' in nombre and 'Completo' in nombre): 
        return "6 Fusión (Completa)"
    return nombre

# -------------------------------------------------------------------------
# 1. EXPLICABILIDAD SHAP 
# -------------------------------------------------------------------------
print("\n Calculando interpretabilidad SHAP (Regresión Logística)...")
try:
    df_shap = df_pacientes.copy()
    
    # SHAP TABULAR COMPLETO (Escenario 4)
    if 'M/F' not in df_shap.columns and 'M/F' in df_demog.columns:
        df_shap = pd.merge(df_shap, df_demog[['id_paciente', 'M/F']], left_on='id', right_on='id_paciente', how='left')
    if 'M/F' in df_shap.columns:
        df_shap['M/F_bin'] = df_shap['M/F'].map({'M': 1, 'F': 0}).fillna(0)
        cols_tab_shap = ['Age', 'Educ', 'SES', 'eTIV', 'M/F_bin', 'MMSE', 'nWBV']
    else:
        cols_tab_shap = ['Age', 'Educ', 'SES', 'eTIV', 'MMSE', 'nWBV']

    y_true_shap = df_shap['etiqueta'].values

    imputer_shap = KNNImputer(n_neighbors=5)
    scaler_shap = StandardScaler()
    X_tab_raw = imputer_shap.fit_transform(df_shap[cols_tab_shap])
    X_tab_sc = scaler_shap.fit_transform(X_tab_raw)

    modelo_tabular_shap = LogisticRegression(penalty='l2', C=0.5, class_weight='balanced', random_state=SEED)
    modelo_tabular_shap.fit(X_tab_sc, y_true_shap)

    explainer_tab = shap.LinearExplainer(modelo_tabular_shap, X_tab_sc)
    shap_values_tab = explainer_tab.shap_values(X_tab_sc)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values_tab, X_tab_sc, feature_names=cols_tab_shap, plot_type="dot", show=False)
    plt.title('SHAP: Impacto de Variables Clínicas (Tabular Completo)', fontweight='bold', fontsize=16, y=1.05)
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_GRAFICOS, '01_SHAP_Tabular.png'), dpi=300, bbox_inches='tight')
    plt.show(); plt.close()
    print(" ✓ Gráfico SHAP Tabular generado.")

except Exception as e:
    print(f" Aviso: No se pudo generar el gráfico SHAP ({e}).")

 
# -------------------------------------------------------------------------
# 2. CURVAS DE APRENDIZAJE Y SWA (PÉRDIDA VISUAL)
# -------------------------------------------------------------------------
try:
    datos_npz = np.load(os.path.join('./modelos_entrenados/', 'resultados_visuales_oof.npz'))
    plt.figure(figsize=(10, 6))
    epocas = range(1, len(datos_npz['train_loss_curve']) + 1)
    plt.plot(epocas, datos_npz['train_loss_curve'], label='Train Loss', color='#2980b9', lw=2.5)
    plt.plot(epocas, datos_npz['val_loss_curve'], label='Validation Loss', color='#c0392b', lw=2.5)
    plt.axvline(x=25, color='#27ae60', linestyle='--', lw=2, label='Activación SWA')
    plt.title('Curva de Aprendizaje Visual (ConvNeXt-Tiny)', fontweight='bold', fontsize=16)
    plt.xlabel('Épocas'); plt.ylabel('Pérdida (Loss)')
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_GRAFICOS, '02_Curva_Loss.png'), dpi=300)
    plt.show(); plt.close()
    print(" ✓ Curva de Loss generada.")
except Exception as e: 
    print(f" Aviso: Loss no generada ({e})")

# -------------------------------------------------------------------------
# 3. CURVAS ROC LISAS (RENDIMIENTO GLOBAL)
# -------------------------------------------------------------------------
plt.figure(figsize=(10, 8))
mean_fpr = np.linspace(0, 1, 100)

for idx, nombre in enumerate(orden_filtrado):
    tprs, aucs = [], []
    for fold_data in almacen_resultados[nombre]:
        if 'fpr' in fold_data and len(fold_data['fpr']) > 2:
            interp_tpr = np.interp(mean_fpr, fold_data['fpr'], fold_data['tpr'])
            interp_tpr[0] = 0.0; tprs.append(interp_tpr); aucs.append(fold_data['auc'])
    if tprs:
        mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0; mean_auc = np.mean(aucs)
        etiqueta_limpia = obtener_etiqueta(nombre)
        plt.plot(mean_fpr, mean_tpr, color=colores_modelos[idx % len(colores_modelos)], lw=2.5, label=f'{etiqueta_limpia} (AUC = {mean_auc:.3f})')

plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='k', label='Línea de Azar')
plt.title('Curvas ROC Comparativas', fontweight='bold', fontsize=16)
plt.xlabel('1 - Especificidad (Tasa de Falsos Positivos)')
plt.ylabel('Sensibilidad (Tasa de Verdaderos Positivos)')
plt.legend(loc="lower right", fontsize='small')
plt.tight_layout()
plt.savefig(os.path.join(DIR_GRAFICOS, '03_Curvas_ROC.png'), dpi=300)
plt.show(); plt.close()
print(" ✓ Curvas ROC generadas.")

# -------------------------------------------------------------------------
# 4. MATRICES DE CONFUSIÓN BINARIAS (ESTUDIO DE ABLACIÓN)
# -------------------------------------------------------------------------
# CUADRÍCULA 2x3 PERFECTA PARA 6 MODELOS
fig, axes = plt.subplots(2, 3, figsize=(18, 11)) 
fig.suptitle('Matrices de Confusión Acumuladas (Out-Of-Fold)', fontsize=22, fontweight='bold', y=1.02)
axes = axes.flatten()

for i, nombre in enumerate(orden_filtrado):
    if almacen_resultados[nombre] and 'cm' in almacen_resultados[nombre][0]:
        cm_global = np.sum([r['cm'] for r in almacen_resultados[nombre]], axis=0)
        sns.heatmap(cm_global, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False, annot_kws={"size": 18, "weight": "bold"})
        etiqueta_limpia = obtener_etiqueta(nombre)
        axes[i].set_title(etiqueta_limpia, fontsize=14, fontweight='bold')
        axes[i].set_xticklabels(['Pred Sano', 'Pred Alzh'], fontsize=12)
        axes[i].set_yticklabels(['Real Sano', 'Real Alzh'], fontsize=12, rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(DIR_GRAFICOS, '04_Matrices_Confusion.png'), dpi=300, bbox_inches='tight')
plt.show(); plt.close()
print(" ✓ Matrices de Confusión generadas.")

# -------------------------------------------------------------------------
# 5. DISTRIBUCIÓN KDE (SEPARABILIDAD TOPOLÓGICA)
# -------------------------------------------------------------------------
# CUADRÍCULA 2x3 PERFECTA PARA 6 MODELOS
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('KDE - Separabilidad de Clases (Sanos vs Alzheimer)', fontsize=22, fontweight='bold', y=1.02)
axes = axes.flatten()

for i, nombre in enumerate(orden_filtrado):
    if almacen_resultados[nombre] and 'y_true' in almacen_resultados[nombre][0]:
        y_true_all = np.concatenate([r['y_true'] for r in almacen_resultados[nombre]])
        y_prob_all = np.concatenate([r['y_prob'] for r in almacen_resultados[nombre]])
        
        scaler_kde = MinMaxScaler()
        y_prob_scaled = scaler_kde.fit_transform(y_prob_all.reshape(-1, 1)).flatten()
        
        sns.kdeplot(y_prob_scaled[y_true_all == 0], color='#3498db', fill=True, alpha=0.5, lw=2, ax=axes[i], label='Sanos')
        sns.kdeplot(y_prob_scaled[y_true_all == 1], color='#e74c3c', fill=True, alpha=0.5, lw=2, ax=axes[i], label='Alzheimer')
        
        axes[i].axvline(x=np.median(y_prob_scaled), color='black', linestyle='--', lw=1.5, alpha=0.7)
        etiqueta_limpia = obtener_etiqueta(nombre)
        axes[i].set_title(etiqueta_limpia, fontsize=14, fontweight='bold')
        axes[i].set_xlim(-0.05, 1.05); axes[i].set_xlabel('Score de Riesgo'); axes[i].set_ylabel('Densidad')
        if i == 0: axes[i].legend(loc='upper right')

plt.tight_layout()
plt.savefig(os.path.join(DIR_GRAFICOS, '05_KDE_Todos_Modelos.png'), dpi=300, bbox_inches='tight')
plt.show(); plt.close()
print(" ✓ Gráficos KDE generados.")

# -------------------------------------------------------------------------
# 6. CURVAS PRECISION-RECALL (PR-AUC)
# -------------------------------------------------------------------------
plt.figure(figsize=(10, 8))
mean_recall = np.linspace(0, 1, 100)

for idx, nombre in enumerate(orden_filtrado):
    precs, pr_aucs = [], []
    for fold_data in almacen_resultados[nombre]:
        if 'recall' in fold_data and len(fold_data['recall']) > 2:
            interp_prec = np.interp(mean_recall, fold_data['recall'][::-1], fold_data['precision'][::-1])
            precs.append(interp_prec); pr_aucs.append(fold_data['pr_auc'])
    if pr_aucs:
        mean_prec = np.mean(precs, axis=0); mean_pr_auc = np.mean(pr_aucs)
        etiqueta_limpia = obtener_etiqueta(nombre)
        plt.plot(mean_recall, mean_prec, color=colores_modelos[idx % len(colores_modelos)], lw=2.5, label=f'{etiqueta_limpia} (PR-AUC = {mean_pr_auc:.3f})')

plt.title('Curvas Precision-Recall', fontweight='bold', fontsize=16)
plt.xlabel('Recall (Sensibilidad)')
plt.ylabel('Precisión (Valor Predictivo Positivo)')
plt.legend(loc="lower left", fontsize='small')
plt.tight_layout()
plt.savefig(os.path.join(DIR_GRAFICOS, '06_Curvas_PR.png'), dpi=300)
plt.show(); plt.close()
print(" ✓ Curvas PR generadas.")

 

In [ ]:
# ==============================================================================
# CELDA 8: INTERPRETABILIDAD BIOLÓGICA ESPACIAL (GRAD-CAM 2x4 ALEATORIO)
# ==============================================================================

DIR_GRAFICOS = './graficos_tfg/'
os.makedirs(DIR_GRAFICOS, exist_ok=True)

 FILAS = 2
COLUMNAS = 4
NUM_PACIENTES = FILAS * COLUMNAS  # Total de 8 pacientes
TOP_K_CORTES = 1  # Nos quedamos con el corte más patológico de cada paciente

# 1. Recuperación del Modelo desde el Disco
modelo_xai = HybridAgile_MIDL().to(dispositivo)
ruta_pesos = os.path.join(DIR_GUARDADO, 'swa_model_fold_0.pth')

try:
    modelo_xai.load_state_dict(torch.load(ruta_pesos, map_location=dispositivo))
    print("Pesos SWA cargados correctamente desde disco.")
except FileNotFoundError:
    print("No se hallaron pesos entrenados. Se usarán pesos aleatorios para la demostración.")

modelo_xai.eval()

# 2. Selección aleatoria
df_ad = df_pacientes[df_pacientes['etiqueta'] == 1].copy()
num_imagenes_reales = min(NUM_PACIENTES, len(df_ad))

df_ad_sample = df_ad.sample(n=num_imagenes_reales).reset_index(drop=True)

ds_xai = OASIS_Visual_Dataset(df_ad_sample, is_train=False)
loader_xai = DataLoader(ds_xai, batch_size=1, num_workers=0, shuffle=False)

# 3. Preparación de los Hooks
activaciones, gradientes = {}, {}
def forward_hook(m, i, o): activaciones['value'] = o
def backward_hook(m, gi, go): gradientes['value'] = go[0]

capa_objetivo = modelo_xai.backbone.features[-1]
handle_fw = capa_objetivo.register_forward_hook(forward_hook)
handle_bw = capa_objetivo.register_full_backward_hook(backward_hook)

overlays_guardados = []
titulos_guardados = []

# 4. Extracción de Mapas Térmicos
for idx, (imgs, targets) in enumerate(loader_xai):
    input_tensor = imgs.to(dispositivo)
    paciente_id = df_ad_sample.iloc[idx]['id']
    
    modelo_xai.zero_grad()
    logits, latents, attention = modelo_xai(input_tensor)
    
    loss = logits[0, 0] 
    loss.backward(retain_graph=False)
    
    feats = activaciones['value'].detach()
    grads = gradientes['value'].detach()
    
    pesos_canales = torch.mean(grads, dim=[2, 3], keepdim=True) 
    cams = torch.sum(pesos_canales * feats, dim=1) 
    cams = F.relu(cams) 
    
    pesos_atencion = attention[0].cpu().detach().numpy()
    
    # Extraer el Top 1
    top_k_indices = np.argsort(pesos_atencion)[-TOP_K_CORTES:][::-1]
    
    for rank, corte_idx in enumerate(top_k_indices):
        cam_actual = cams[corte_idx].cpu().numpy()
        cam_actual = cv2.resize(cam_actual, (224, 224))
        
        if np.max(cam_actual) - np.min(cam_actual) > 0:
            cam_actual = (cam_actual - np.min(cam_actual)) / (np.max(cam_actual) - np.min(cam_actual))
        else:
            cam_actual = np.zeros_like(cam_actual)
        
        img_original = input_tensor[0, corte_idx].cpu().numpy().transpose(1, 2, 0)
        img_original = (img_original - img_original.min()) / (img_original.max() - img_original.min() + 1e-8)
        
        heatmap = cv2.applyColorMap(np.uint8(255 * cam_actual), cv2.COLORMAP_JET)
        heatmap = np.float32(heatmap) / 255
        heatmap = heatmap[:, :, ::-1] 
        
        cam_overlay = heatmap * 0.4 + img_original * 0.6
        cam_overlay = cam_overlay / np.max(cam_overlay)
        
        overlays_guardados.append(cam_overlay)
        titulos_guardados.append(f"ID: {paciente_id}\nCorte: {corte_idx+1}/16 | Atn: {pesos_atencion[corte_idx]:.2f}")

handle_fw.remove()
handle_bw.remove()

# 5. Cuadricula
fig, axes = plt.subplots(FILAS, COLUMNAS, figsize=(20, 10))
fig.suptitle('8. Focos de Neurodegeneración: Máxima Atención en 8 Pacientes Aleatorios', 
             fontweight='bold', fontsize=20, y=1.02)

axes_flat = axes.flatten()

for i, ax in enumerate(axes_flat):
    if i < len(overlays_guardados):
        ax.imshow(overlays_guardados[i])
        ax.set_title(titulos_guardados[i], fontsize=13, fontweight='bold', pad=8)
        ax.axis('off')
    else:
        # Por si el dataset se queda sin imágenes por algún motivo
        ax.axis('off')

plt.tight_layout()
ruta_guardado = os.path.join(DIR_GRAFICOS, '08_Grad_CAM_2x4_Images.png')
plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
plt.show()

 

In [ ]:
# ==============================================================================
# CELDA 8B: ESTUDIO DE CASO "EFECTO TECHO" (CORTES ALEATORIOS)
# ==============================================================================



DIR_GRAFICOS = './graficos_tfg_final/'
os.makedirs(DIR_GRAFICOS, exist_ok=True)

# CONTROL DE LA CUADRÍCULA (1x5 para que cuadre con tu tabla)
FILAS = 1
COLUMNAS = 5 

# 1. Recuperación Autónoma del Modelo desde el Disco
modelo_xai = HybridAgile_MIDL().to(dispositivo)
ruta_pesos = os.path.join('./modelos_entrenados/', 'swa_model_fold_0.pth')

try:
    modelo_xai.load_state_dict(torch.load(ruta_pesos, map_location=dispositivo))
    print(" ✓ Pesos SWA cargados correctamente desde disco para el Estudio de Caso.")
except FileNotFoundError:
    print(" ❌ No se hallaron pesos entrenados. Se usarán pesos aleatorios para la demostración.")

modelo_xai.eval()

# 2. SELECCIÓN DE PACIENTES (LOS 5 DEL EFECTO TECHO)
pacientes_techo = ['OAS1_0042', 'OAS1_0205', 'OAS1_0060', 'OAS1_0263', 'OAS1_0304']

# Filtramos solo esos pacientes
df_techo = df_pacientes[df_pacientes['id'].isin(pacientes_techo)].copy()

# Ordenamos el dataframe para que salgan EXACTAMENTE en el orden de tu tabla
df_techo['id_cat'] = pd.Categorical(df_techo['id'], categories=pacientes_techo, ordered=True)
df_techo = df_techo.sort_values('id_cat').reset_index(drop=True)

# Creamos el Dataset y DataLoader solo para estos 5
ds_xai = OASIS_Visual_Dataset(df_techo, is_train=False)
loader_xai = DataLoader(ds_xai, batch_size=1, num_workers=0, shuffle=False)

# 3. Preparación de los Hooks para Grad-CAM
activaciones, gradientes = {}, {}
def forward_hook(m, i, o): activaciones['value'] = o
def backward_hook(m, gi, go): gradientes['value'] = go[0]

capa_objetivo = modelo_xai.backbone.features[-1]
handle_fw = capa_objetivo.register_forward_hook(forward_hook)
handle_bw = capa_objetivo.register_full_backward_hook(backward_hook)

overlays_guardados = []
titulos_guardados = []

# 4. Extracción de Mapas Térmicos
for idx, (imgs, targets) in enumerate(loader_xai):
    input_tensor = imgs.to(dispositivo)
    paciente_id = df_techo.iloc[idx]['id']
    
    # Intentamos extraer el MMSE original si está en el dataframe para el título
    mmse_val = df_techo.iloc[idx]['MMSE'] if 'MMSE' in df_techo.columns else "Alta"
    
    modelo_xai.zero_grad()
    logits, latents, attention = modelo_xai(input_tensor)
    
    loss = logits[0, 0] 
    loss.backward(retain_graph=False)
    
    feats = activaciones['value'].detach()
    grads = gradientes['value'].detach()
    
    pesos_canales = torch.mean(grads, dim=[2, 3], keepdim=True) 
    cams = torch.sum(pesos_canales * feats, dim=1) 
    cams = F.relu(cams) 
    
    pesos_atencion = attention[0].cpu().detach().numpy()
    
    # -----------------------------------------------------------------
    # MODIFICACIÓN: Selección de 1 corte completamente al azar (0 a 15)
    # -----------------------------------------------------------------
    corte_aleatorio = random.randint(0, 15)
    top_k_indices = [corte_aleatorio]
    
    for rank, corte_idx in enumerate(top_k_indices):
        cam_actual = cams[corte_idx].cpu().numpy()
        cam_actual = cv2.resize(cam_actual, (224, 224))
        
        if np.max(cam_actual) - np.min(cam_actual) > 0:
            cam_actual = (cam_actual - np.min(cam_actual)) / (np.max(cam_actual) - np.min(cam_actual))
        else:
            cam_actual = np.zeros_like(cam_actual)
        
        img_original = input_tensor[0, corte_idx].cpu().numpy().transpose(1, 2, 0)
        img_original = (img_original - img_original.min()) / (img_original.max() - img_original.min() + 1e-8)
        
        heatmap = cv2.applyColorMap(np.uint8(255 * cam_actual), cv2.COLORMAP_JET)
        heatmap = np.float32(heatmap) / 255
        heatmap = heatmap[:, :, ::-1] 
        
        cam_overlay = heatmap * 0.4 + img_original * 0.6
        cam_overlay = cam_overlay / np.max(cam_overlay)
        
        overlays_guardados.append(cam_overlay)
        # Añadimos la advertencia de "Aleatorio" al título
        titulos_guardados.append(f"ID: {paciente_id}\nCorte: {corte_idx+1}/16 | MMSE: {mmse_val}")

handle_fw.remove()
handle_bw.remove()

# 5. DIBUJO DE LA CUADRÍCULA (GRID 1x5 ESTRICTO)
fig, axes = plt.subplots(FILAS, COLUMNAS, figsize=(22, 5)) 
fig.suptitle('Estudio de Caso: Cortes Aleatorios (Cuidado, no representa el foco real de la red)', 
             fontweight='bold', fontsize=18, y=1.1)

axes_flat = axes.flatten()

for i, ax in enumerate(axes_flat):
    if i < len(overlays_guardados):
        ax.imshow(overlays_guardados[i])
        ax.set_title(titulos_guardados[i], fontsize=12, fontweight='bold', pad=8)
        ax.axis('off')
    else:
        ax.axis('off')

plt.tight_layout()
ruta_guardado = os.path.join(DIR_GRAFICOS, '09_Grad_CAM_Efecto_Techo_ALEATORIO.png')
plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
plt.show()

 